In [ ]:
import numpy as np
from learn_s_hat_toy import make_srs
import copy
from gould_2026.estimator import ArrayWithTime
from gould_2026.plotting import Palette
import matplotlib.pyplot as plt
import pandas as pd
from enum import Enum
from pathlib import Path
from typing import Literal
import scipy

In [ ]:
UFunctionPresets = Literal[
    'curvy',
    'curvy_flips',
    'curvy_spins',
    'curvy_alld_resp'
]

In [ ]:
output: Path | None = None
u_function: UFunctionPresets = 'curvy_alld_resp'
n_runs = 5
n_rotations = 100
transition_time = 40

In [ ]:
output = Path(output).resolve() if output is not None else None

In [ ]:
from gould_2026.save_to_cache import save_to_cache

@save_to_cache('learn_s_hat_toy_1step_spin', location='/mnt/data/gould_2026_cache/')
def f(u_function: UFunctionPresets='curvy_flips', n_runs=100, n_rotations=60, transition_time=30):
    rng = np.random.default_rng(0)
    srs = make_srs(copy.deepcopy(rng), n_runs=n_runs, u_function=u_function, show_tqdm=True, n_rotations=n_rotations, transition_time=transition_time)
    return rng, srs

rng, srs = f(u_function=u_function, n_runs=n_runs, n_rotations=n_rotations, transition_time=transition_time)


In [ ]:
%matplotlib inline
def square_err_array(errs):
    lengths = [len(e.t) for e in errs]
    common_length = min(lengths)
    longest_idx = np.argmax(lengths)
    # a = [np.linalg.norm(e.slice(slice(-common_length, None))**2, axis=1) for e in errs]
    # t = errs[0].slice(slice(-common_length, None)).t
    # return ArrayWithTime(a, t)

    max_length = max(lengths)

    assert np.std(np.vstack([e.t[-common_length:] for e in errs]), axis=0).max() < 1e-10

    filled_errors = []
    for e in errs:
        early_nans = np.array([e[0] * np.nan] * (max_length - len(e))).reshape(-1,3)
        padded_e = np.vstack([np.squeeze(early_nans), e])
        filled_errors.append(padded_e)


    a = np.squeeze([np.linalg.norm(e, axis=1) for e in filled_errors])**2
    t = errs[longest_idx].t
    good_idx = np.nonzero(np.isnan(a).sum(axis=0) < int(a.shape[0] * .75))[0][0]
    good_idx = 10
    ret = ArrayWithTime(a[:,good_idx:], t[good_idx:])
    return ret


# fig, ax = plt.subplots(figsize=np.array((17,5))/1.2, layout='constrained')
fig, ax = plt.subplots(figsize=np.array([3.15,.945])*2, layout='constrained')

ax2 = ax.twinx()

kernel = np.hstack([np.zeros(50), np.ones(51)])
# kernel = np.ones(51)
kernel = kernel / kernel.sum()
time_kernel = kernel * 0
time_kernel[len(kernel)//2] = 1

errs = square_err_array([sr.log['pred_error'] for sr in srs['unaware of stim']])
mean_errors_unaware = np.nanmean(errs, axis=0)
ax.plot(errs.t, errs[0], label='unaware (single trial)', color=Palette.blind, alpha=.1,)
smoothed_mean_errs_unaware = ArrayWithTime(np.convolve(mean_errors_unaware, kernel, 'valid'), np.convolve(errs.t, time_kernel, 'valid'))
ax2.plot(smoothed_mean_errs_unaware.t, smoothed_mean_errs_unaware, label='unaware (averaged, smoothed)', color=Palette.blind)
errs_unaware = errs

errs = square_err_array([sr.log['pred_error'] for sr in srs['learning from stim']])
mean_errors_aware = np.nanmean(errs, axis=0)
ax.plot(errs.t, errs[0], label='aware (single trial)', color=Palette.stim_regressed, alpha=.1)
smoothed_mean_errs_aware = ArrayWithTime(np.convolve(mean_errors_aware, kernel, 'valid'), np.convolve(errs.t, time_kernel, 'valid'))
ax2.plot(smoothed_mean_errs_aware.t, smoothed_mean_errs_aware, label='aware (averaged, smoothed)', color=Palette.stim_regressed)
errs_aware = errs

ax2.axvline(transition_time, color='gray', linestyle='--')

ax.set_xlabel('Time (s)')
ax.set_ylabel('error')
ax2.set_ylabel('averaged error')
ax2.legend(['Blind','Stim. Reg.'],loc='upper right')
# ax.legend(loc='upper left')
ax.set_xlim(0,n_rotations)
ax2.set_xlim(0,n_rotations)

ax.set_ylim([0, 60])
ax2.set_ylim([0,6])

ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.set_yticks([0, 20, 40, 60])
ax2.set_yticks([0, 2, 4, 6])

# x = np.linspace(0,50,200)
# ax2.plot(x+50,  % (2 * np.pi))

if output is not None:
    fig.savefig(output, pad_inches=0)


In [ ]:
s = ~np.isnan(errs_aware)
l = lambda x: x.T.slice_by_time(slice(75, None)).flatten()

test_result = scipy.stats.wilcoxon(l(errs_aware), l(errs_unaware))

if output is not None:
    with open(output.with_suffix('.txt'), 'w') as f:
        f.write(f'{test_result}')


In [ ]:

df = pd.DataFrame([
    {'condition': 'unaware of stim', 'errors': mean_errors_unaware},
    {'condition': 'learning from stim', 'errors': mean_errors_aware},
])

df['overall_err_mean'] = df['errors'].apply(lambda x: np.nanmean(x.slice_by_time(slice(None,None))))
df['overall_err_std'] = df['errors'].apply(lambda x: np.nanstd(x.slice_by_time(slice(None,None))))
df['last_rotation_err_mean'] = df['errors'].apply(lambda x: np.nanmean(x.slice_by_time(slice(75,None))))
df['last_rotation_err_std'] = df['errors'].apply(lambda x: np.nanstd(x.slice_by_time(slice(75,None))))
df['initial_fit_err_mean'] = df['errors'].apply(lambda x: np.nanmean(x.slice_by_time(slice(None,25))))
df['initial_fit_err_std'] = df['errors'].apply(lambda x: np.nanstd(x.slice_by_time(slice(None,25))))
del df['errors']

df


In [ ]:
fig, ax = plt.subplots()
for sr in srs['learning from stim']:
    sr.stim_reg.plot_length_scales(ax)

In [ ]:
%matplotlib inline


fig, ax = plt.subplots(figsize=np.array((17,5))/1.2, layout='constrained')

differences:ArrayWithTime = errs_aware.T - errs_unaware[:,0:].T

samples = []
for _ in range(1000):
    bootstrap_sample = rng.choice(differences, axis=1, size=differences.shape[1], replace=True)
    t = errs_aware.t
    jittered_t = t + rng.uniform(-1,1,size=t.shape) * 1
    shuffled_order = np.argsort(jittered_t)
    samples.append(np.nanmean(bootstrap_sample[shuffled_order], axis=1))

samples = np.array(samples).T

lower, upper = np.quantile(samples, [0.005, 0.995], axis=1)
ax.fill_between(x=t, y1=lower, y2=upper, color='C0')
ax.axhline(0, color='gray', linestyle='--')



def find(x):
    return np.nonzero(x)[0]

x = find(np.hstack([upper, [np.inf]]) > 0)
intervals = []
for z in find(np.diff(x) > 500):
    interval = (x[z] + 1, x[z+1] - 1)
    ax.axvline(t[interval[0]], color='k', alpha=.5)
    ax.axvline(t[interval[1]], color='k', alpha=.5, linestyle='--')
    intervals.append(interval)



In [ ]:
interval = intervals[0]
d = differences[interval[0]]
d = d[~np.isnan(d)]
scipy.stats.wilcoxon(d)

In [ ]:
def ratio(t_idx):
    print(np.nanmean(errs_aware.T.slice_by_time(t_idx))/np.nanmean(errs_unaware.T.slice_by_time(t_idx)))

print((idx:=t[interval[0]]), np.nanmean(errs_aware.T.slice_by_time(idx)), np.nanmean(errs_unaware.T.slice_by_time(idx)))
ratio(idx)

idx = transition_time + 10
ratio(idx)


In [ ]:
df = pd.DataFrame()
for group, v in srs.items():
    for sr_i, sr in enumerate(v):
        stim_ts = sr.log['stim_intended_samples'].t
        errors = sr.log['pred_error'].slice_by_time(stim_ts)

        df = pd.concat([df, pd.DataFrame({
            'raw_error': [e for e in errors],
            't': errors.t,
            'sr_i': sr_i,
            'group': group
        })])

df['error'] = df['raw_error'].apply(np.mean)

df.dropna(inplace=True)